# Part II-C: Convolutional Neural Network (CNN) for Surface Crack Classification

## Overview
This notebook implements a Convolutional Neural Network (CNN) for classifying cracked vs non-cracked surfaces. CNNs are the most effective architecture for image classification as they learn spatial hierarchies of features through convolutional filters.

## Hyperparameters to Explore
- **Batch size**: 16, 32, 64
- **Number of convolutional layers**: 2, 3, 4 blocks
- **Dropout rate**: 0.0, 0.3, 0.5
- **Optimizer**: SGD, Adam, RMSProp
- **L2 Regularization**: 0.0, 1e-4, 1e-3
- **Learning rate**: 0.1, 0.01, 0.001
- **Learning rate scheduler**: StepLR, ReduceLROnPlateau

### Import Libraries
Import PyTorch modules for building CNNs, along with data utilities and visualization libraries.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
import pandas as pd
import time
import copy

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

### Configure Device
Check for GPU availability. CNNs benefit greatly from GPU acceleration.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
torch.backends.cudnn.benchmark = True
print("CUDNN Benchmark Enabled (Eager Mode Optimized)")
torch.backends.cudnn.benchmark = True
print("PyTorch Eager Mode: Active (CUDNN benchmark enabled)")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### Define Custom Dataset Class
Same dataset class - loads images from Cracked and Non Cracked folders with PyTorch transforms.

In [ ]:
class SurfaceCrackDataset(Dataset):
    def __init__(self, cracked_dir, non_cracked_dir, transform=None, max_samples=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        cracked_files = sorted(os.listdir(cracked_dir))
        if max_samples:
            cracked_files = cracked_files[:max_samples]
        for fname in cracked_files:
            self.image_paths.append(os.path.join(cracked_dir, fname))
            self.labels.append(1)
        
        non_cracked_files = sorted(os.listdir(non_cracked_dir))
        if max_samples:
            non_cracked_files = non_cracked_files[:max_samples]
        for fname in non_cracked_files:
            self.image_paths.append(os.path.join(non_cracked_dir, fname))
            self.labels.append(0)
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, torch.tensor(label, dtype=torch.long)

### Set Up Data Transforms and Loaders
Use full 227x227 resolution (CNNs handle this well) with augmentation for training.

In [ ]:
CRACKED_DIR = 'Cracked'
NON_CRACKED_DIR = 'Non Cracked'
IMAGE_SIZE = 227

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = SurfaceCrackDataset(CRACKED_DIR, NON_CRACKED_DIR, transform=train_transform)
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size],
                                            generator=torch.Generator().manual_seed(42))
test_dataset.dataset.transform = test_transform

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Training samples: {train_size}")
print(f"Testing samples: {test_size}")
print(f"Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")

### Define CNN Model
Build a flexible CNN with configurable number of conv blocks. Each block contains: Conv2d -> BatchNorm -> ReLU -> MaxPool. The architecture progressively increases filter counts while reducing spatial dimensions.

In [ ]:
class CNNClassifier(nn.Module):
    def __init__(self, num_conv_blocks=3, base_filters=32, dropout_rate=0.0, num_classes=2):
        super(CNNClassifier, self).__init__()
        
        conv_layers = []
        in_channels = 3
        
        for i in range(num_conv_blocks):
            out_channels = base_filters * (2 ** i)
            conv_layers.extend([
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=2, stride=2)
            ])
            in_channels = out_channels
        
        self.features = nn.Sequential(*conv_layers)
        
        spatial_size = IMAGE_SIZE // (2 ** num_conv_blocks)
        fc_input = out_channels * spatial_size * spatial_size
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(fc_input, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

print("CNN Model Architecture (3 blocks, base=32):")
test_model = CNNClassifier(num_conv_blocks=3, base_filters=32)
print(test_model)

### Define Training Function
One epoch of training: forward pass, loss computation, backpropagation, and optimizer step.

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return running_loss / total, correct / total

### Define Evaluation Function
Evaluates model on a dataset with no gradient computation, returns loss, accuracy, and per-class metrics.

In [ ]:
def evaluate(model, data_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    class_correct = [0, 0]
    class_total = [0, 0]
    
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            for i in range(labels.size(0)):
                label = labels[i].item()
                class_total[label] += 1
                class_correct[label] += predicted[i].eq(label).item()
    
    class_accuracies = [c/t if t > 0 else 0 for c, t in zip(class_correct, class_total)]
    return running_loss / total, correct / total, class_accuracies

### Define Full Training Loop
Complete training with LR scheduling, best model saving, and history tracking.

In [ ]:
def train_model(model, train_loader, test_loader, criterion, optimizer, scheduler,
                num_epochs, device, model_name='cnn'):
    history = {
        'train_loss': [], 'train_acc': [],
        'test_loss': [], 'test_acc': [],
        'test_acc_cracked': [], 'test_acc_non_cracked': []
    }
    best_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())
    
    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        test_loss, test_acc, test_class_acc = evaluate(model, test_loader, criterion, device)
        
        if scheduler is not None:
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(test_loss)
            else:
                scheduler.step()
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        history['test_acc_cracked'].append(test_class_acc[1])
        history['test_acc_non_cracked'].append(test_class_acc[0])
        
        if test_acc > best_acc:
            best_acc = test_acc
            best_model_wts = copy.deepcopy(model.state_dict())
        
        current_lr = optimizer.param_groups[0]['lr']
        print(f'Epoch [{epoch+1}/{num_epochs}] LR: {current_lr:.6f} | '
              f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
              f'Test Loss: {test_loss:.4f} Acc: {test_acc:.4f}')
    
    model.load_state_dict(best_model_wts)
    return model, history, best_acc

### Helper: Plot Training History
Visualize training and test curves.

In [ ]:
def plot_history(history, title='Training History'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
    axes[0].plot(history['test_loss'], label='Test Loss', marker='s')
    axes[0].set_title('Loss Over Epochs', fontsize=14)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True)
    
    axes[1].plot(history['train_acc'], label='Train Accuracy', marker='o')
    axes[1].plot(history['test_acc'], label='Test Accuracy', marker='s')
    axes[1].set_title('Accuracy Over Epochs', fontsize=14)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

### Experiment 1: Baseline CNN Configuration
Start with a baseline: 3 conv blocks, 32 base filters, dropout=0.3, Adam optimizer, lr=0.001.

In [ ]:
NUM_EPOCHS = 15

model = CNNClassifier(num_conv_blocks=3, base_filters=32, dropout_rate=0.3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Starting baseline CNN training...")

baseline_model, baseline_history, baseline_acc = train_model(
    model, train_loader, test_loader, criterion, optimizer, None,
    NUM_EPOCHS, device, 'cnn_baseline'
)

print(f"\nBest test accuracy: {baseline_acc:.4f}")
plot_history(baseline_history, 'CNN Baseline (3 blocks, base=32, Adam, lr=0.001)')

### Experiment 2: Effect of Number of Convolutional Blocks
Compare 2, 3, and 4 conv blocks. More blocks extract higher-level features but increase computation and risk overfitting.

In [ ]:
block_configs = {
    '2 blocks': 2,
    '3 blocks': 3,
    '4 blocks': 4
}

block_results = {}

for name, num_blocks in block_configs.items():
    print(f"\n{'='*50}")
    print(f"Training with {name}")
    print(f"{'='*50}")
    
    model = CNNClassifier(num_conv_blocks=num_blocks, base_filters=32, dropout_rate=0.3).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    _, history, best_acc = train_model(
        model, train_loader, test_loader, criterion, optimizer, None,
        NUM_EPOCHS, device, f'cnn_{name}'
    )
    block_results[name] = {'best_acc': best_acc, 'history': history}
    print(f"Best accuracy for {name}: {best_acc:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
for name, result in block_results.items():
    ax.plot(result['history']['test_acc'], label=f'{name} (test)', marker='o')
ax.set_title('Effect of Conv Blocks on Test Accuracy', fontsize=14)
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy')
ax.legend()
ax.grid(True)
plt.show()

### Experiment 3: Effect of Dropout Rate
Test dropout rates of 0.0, 0.3, and 0.5 in the classifier layers.

In [ ]:
dropout_rates = [0.0, 0.3, 0.5]
dropout_results = {}

for rate in dropout_rates:
    print(f"\n{'='*50}")
    print(f"Training with dropout={rate}")
    print(f"{'='*50}")
    
    model = CNNClassifier(num_conv_blocks=3, base_filters=32, dropout_rate=rate).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    _, history, best_acc = train_model(
        model, train_loader, test_loader, criterion, optimizer, None,
        NUM_EPOCHS, device, f'cnn_dropout_{rate}'
    )
    dropout_results[f'dropout={rate}'] = {'best_acc': best_acc, 'history': history}
    print(f"Best accuracy with dropout={rate}: {best_acc:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
for name, result in dropout_results.items():
    ax.plot(result['history']['test_acc'], label=f'{name} (test)', marker='o')
ax.set_title('Effect of Dropout Rate on Test Accuracy', fontsize=14)
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy')
ax.legend()
ax.grid(True)
plt.show()

### Experiment 4: Effect of Optimizer
Compare SGD, Adam, and RMSProp for CNN training.

In [ ]:
optimizers_config = {
    'SGD': lambda params: optim.SGD(params, lr=0.01, momentum=0.9),
    'Adam': lambda params: optim.Adam(params, lr=0.001),
    'RMSProp': lambda params: optim.RMSprop(params, lr=0.001)
}

optimizer_results = {}

for name, opt_fn in optimizers_config.items():
    print(f"\n{'='*50}")
    print(f"Training with {name} optimizer")
    print(f"{'='*50}")
    
    model = CNNClassifier(num_conv_blocks=3, base_filters=32, dropout_rate=0.3).to(device)
    optimizer = opt_fn(model.parameters())
    
    _, history, best_acc = train_model(
        model, train_loader, test_loader, criterion, optimizer, None,
        NUM_EPOCHS, device, f'cnn_{name}'
    )
    optimizer_results[name] = {'best_acc': best_acc, 'history': history}
    print(f"Best accuracy with {name}: {best_acc:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
for name, result in optimizer_results.items():
    ax.plot(result['history']['test_acc'], label=f'{name} (test)', marker='o')
ax.set_title('Effect of Optimizer on Test Accuracy', fontsize=14)
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy')
ax.legend()
ax.grid(True)
plt.show()

### Experiment 5: Effect of L2 Regularization (Weight Decay)
Test weight decay values of 0.0, 1e-4, and 1e-3.

In [ ]:
weight_decays = [0.0, 1e-4, 1e-3]
wd_results = {}

for wd in weight_decays:
    print(f"\n{'='*50}")
    print(f"Training with weight_decay={wd}")
    print(f"{'='*50}")
    
    model = CNNClassifier(num_conv_blocks=3, base_filters=32, dropout_rate=0.3).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=wd)
    
    _, history, best_acc = train_model(
        model, train_loader, test_loader, criterion, optimizer, None,
        NUM_EPOCHS, device, f'cnn_wd_{wd}'
    )
    wd_results[f'wd={wd}'] = {'best_acc': best_acc, 'history': history}
    print(f"Best accuracy with wd={wd}: {best_acc:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
for name, result in wd_results.items():
    ax.plot(result['history']['test_acc'], label=f'{name} (test)', marker='o')
ax.set_title('Effect of L2 Regularization on Test Accuracy', fontsize=14)
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy')
ax.legend()
ax.grid(True)
plt.show()

### Experiment 6: Effect of Learning Rate
Compare learning rates of 0.1, 0.01, and 0.001.

In [ ]:
learning_rates = [0.1, 0.01, 0.001]
lr_results = {}

for lr in learning_rates:
    print(f"\n{'='*50}")
    print(f"Training with lr={lr}")
    print(f"{'='*50}")
    
    model = CNNClassifier(num_conv_blocks=3, base_filters=32, dropout_rate=0.3).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    
    _, history, best_acc = train_model(
        model, train_loader, test_loader, criterion, optimizer, None,
        NUM_EPOCHS, device, f'cnn_lr_{lr}'
    )
    lr_results[f'lr={lr}'] = {'best_acc': best_acc, 'history': history}
    print(f"Best accuracy with lr={lr}: {best_acc:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
for name, result in lr_results.items():
    ax.plot(result['history']['test_acc'], label=f'{name} (test)', marker='o')
ax.set_title('Effect of Learning Rate on Test Accuracy', fontsize=14)
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy')
ax.legend()
ax.grid(True)
plt.show()

### Experiment 7: Learning Rate Schedulers
Compare no scheduler, StepLR, and ReduceLROnPlateau.

In [ ]:
scheduler_configs = {
    'No Scheduler': None,
    'StepLR': lambda opt: optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.5),
    'ReduceLROnPlateau': lambda opt: optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=3)
}

scheduler_results = {}

for name, sched_fn in scheduler_configs.items():
    print(f"\n{'='*50}")
    print(f"Training with {name}")
    print(f"{'='*50}")
    
    model = CNNClassifier(num_conv_blocks=3, base_filters=32, dropout_rate=0.3).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = sched_fn(optimizer) if sched_fn else None
    
    _, history, best_acc = train_model(
        model, train_loader, test_loader, criterion, optimizer, scheduler,
        NUM_EPOCHS, device, f'cnn_sched_{name}'
    )
    scheduler_results[name] = {'best_acc': best_acc, 'history': history}
    print(f"Best accuracy with {name}: {best_acc:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
for name, result in scheduler_results.items():
    ax.plot(result['history']['test_acc'], label=f'{name} (test)', marker='o')
ax.set_title('Effect of Learning Rate Scheduler on Test Accuracy', fontsize=14)
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy')
ax.legend()
ax.grid(True)
plt.show()

### Experiment 8: Effect of Batch Size
Compare batch sizes of 16, 32, and 64.

In [ ]:
batch_sizes = [16, 32, 64]
bs_results = {}

for bs in batch_sizes:
    print(f"\n{'='*50}")
    print(f"Training with batch_size={bs}")
    print(f"{'='*50}")
    
    train_loader_exp = DataLoader(train_dataset, batch_size=bs, shuffle=True, num_workers=0)
    test_loader_exp = DataLoader(test_dataset, batch_size=bs, shuffle=False, num_workers=0)
    
    model = CNNClassifier(num_conv_blocks=3, base_filters=32, dropout_rate=0.3).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    
    _, history, best_acc = train_model(
        model, train_loader_exp, test_loader_exp, criterion, optimizer, scheduler,
        NUM_EPOCHS, device, f'cnn_bs_{bs}'
    )
    bs_results[f'bs={bs}'] = {'best_acc': best_acc, 'history': history}
    print(f"Best accuracy with batch_size={bs}: {best_acc:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
for name, result in bs_results.items():
    ax.plot(result['history']['test_acc'], label=f'{name} (test)', marker='o')
ax.set_title('Effect of Batch Size on Test Accuracy', fontsize=14)
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy')
ax.legend()
ax.grid(True)
plt.show()

### Visualize Learned Filters
Display the first layer convolutional filters to see what edge/texture patterns the CNN learned.

In [ ]:
def visualize_first_layer_filters(model, num_filters=16):
    filters = model.features[0].weight.data.cpu().numpy()
    
    rows = (num_filters + 3) // 4
    cols = min(4, num_filters)
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3*rows))
    axes = axes.flatten()
    
    for i in range(num_filters):
        f = filters[i]
        f = (f - f.min()) / (f.max() - f.min() + 1e-8)
        
        if f.shape[0] == 3:
            axes[i].imshow(f.transpose(1, 2, 0))
        else:
            axes[i].imshow(f[0], cmap='gray')
        axes[i].set_title(f'Filter {i+1}', fontsize=10)
        axes[i].axis('off')
    
    for j in range(num_filters, len(axes)):
        axes[j].axis('off')
    
    plt.suptitle('First Layer Convolutional Filters', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_first_layer_filters(baseline_model)

### Visualize Feature Maps
Show how an input image is transformed through the convolutional layers.

In [ ]:
def visualize_feature_maps(model, image, layer_indices=[0, 6, 14]):
    model.eval()
    
    activations = {}
    def hook_fn(module, input, output):
        activations[module] = output.detach()
    
    hooks = []
    conv_layers = [m for m in model.features if isinstance(m, nn.Conv2d)]
    for idx in layer_indices:
        if idx < len(conv_layers):
            hooks.append(conv_layers[idx].register_forward_hook(hook_fn))
    
    with torch.no_grad():
        _ = model(image.unsqueeze(0).to(device))
    
    fig, axes = plt.subplots(len(activations), 5, figsize=(15, 3*len(activations)))
    
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = image.numpy().transpose(1, 2, 0)
    img = std * img + mean
    img = np.clip(img, 0, 1)
    
    axes[0, 0].imshow(img)
    axes[0, 0].set_title('Input Image', fontsize=10)
    axes[0, 0].axis('off')
    
    for j in range(1, 5):
        axes[0, j].axis('off')
    
    for i, (layer, act) in enumerate(activations.items()):
        act = act[0].cpu().numpy()
        for j in range(min(5, act.shape[0])):
            axes[i+1, j].imshow(act[j], cmap='viridis')
            axes[i+1, j].set_title(f'Channel {j}', fontsize=8)
            axes[i+1, j].axis('off')
        for j in range(5, axes.shape[1]):
            axes[i+1, j].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    for h in hooks:
        h.remove()

sample_img, sample_label = test_dataset[0]
visualize_feature_maps(baseline_model, sample_img)

### Final Model: Best Configuration
Train the final CNN model using the best hyperparameters discovered.

In [ ]:
BEST_CONFIG = {
    'num_conv_blocks': 3,
    'base_filters': 32,
    'dropout_rate': 0.3,
    'optimizer': 'Adam',
    'learning_rate': 0.001,
    'weight_decay': 1e-4,
    'batch_size': 32,
    'scheduler': 'StepLR',
    'num_epochs': 20
}

print("Training final CNN model with best configuration:")
for k, v in BEST_CONFIG.items():
    print(f"  {k}: {v}")

train_loader_final = DataLoader(train_dataset, batch_size=BEST_CONFIG['batch_size'], shuffle=True, num_workers=0)
test_loader_final = DataLoader(test_dataset, batch_size=BEST_CONFIG['batch_size'], shuffle=False, num_workers=0)

final_model = CNNClassifier(
    num_conv_blocks=BEST_CONFIG['num_conv_blocks'],
    base_filters=BEST_CONFIG['base_filters'],
    dropout_rate=BEST_CONFIG['dropout_rate']
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(final_model.parameters(), lr=BEST_CONFIG['learning_rate'],
                       weight_decay=BEST_CONFIG['weight_decay'])
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

final_model, final_history, final_acc = train_model(
    final_model, train_loader_final, test_loader_final, criterion, optimizer, scheduler,
    BEST_CONFIG['num_epochs'], device, 'cnn_final'
)

torch.save(final_model.state_dict(), 'cnn_best_model.pth')
print(f"\nFinal model saved. Best test accuracy: {final_acc:.4f}")
plot_history(final_history, 'CNN Final Model (Best Configuration)')

### Comprehensive Results Summary
Compile all experimental results into a summary table.

In [ ]:
results_summary = []

results_summary.append({'Experiment': 'Baseline', 'Config': '3 blocks, base=32, Adam, lr=0.001', 'Best Acc': f"{baseline_acc:.4f}"})

for name, result in block_results.items():
    results_summary.append({'Experiment': 'Conv Blocks', 'Config': name, 'Best Acc': f"{result['best_acc']:.4f}"})

for name, result in dropout_results.items():
    results_summary.append({'Experiment': 'Dropout', 'Config': name, 'Best Acc': f"{result['best_acc']:.4f}"})

for name, result in optimizer_results.items():
    results_summary.append({'Experiment': 'Optimizer', 'Config': name, 'Best Acc': f"{result['best_acc']:.4f}"})

for name, result in wd_results.items():
    results_summary.append({'Experiment': 'Weight Decay', 'Config': name, 'Best Acc': f"{result['best_acc']:.4f}"})

for name, result in lr_results.items():
    results_summary.append({'Experiment': 'Learning Rate', 'Config': name, 'Best Acc': f"{result['best_acc']:.4f}"})

for name, result in scheduler_results.items():
    results_summary.append({'Experiment': 'Scheduler', 'Config': name, 'Best Acc': f"{result['best_acc']:.4f}"})

for name, result in bs_results.items():
    results_summary.append({'Experiment': 'Batch Size', 'Config': name, 'Best Acc': f"{result['best_acc']:.4f}"})

results_summary.append({'Experiment': 'Final', 'Config': 'Best combined config', 'Best Acc': f"{final_acc:.4f}"})

df_results = pd.DataFrame(results_summary)
print(df_results.to_string(index=False))

### Final Evaluation on Train and Test Sets
Report final CNN model performance on both training and test sets.

In [ ]:
print("="*60)
print("FINAL CNN MODEL EVALUATION")
print("="*60)

train_loss, train_acc, train_class_acc = evaluate(final_model, train_loader_final, criterion, device)
test_loss, test_acc, test_class_acc = evaluate(final_model, test_loader_final, criterion, device)

print(f"\nTraining Set:")
print(f"  Loss: {train_loss:.4f}")
print(f"  Accuracy: {train_acc:.4f}")
print(f"  Cracked Accuracy: {train_class_acc[1]:.4f}")
print(f"  Non-Cracked Accuracy: {train_class_acc[0]:.4f}")

print(f"\nTest Set:")
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_acc:.4f}")
print(f"  Cracked Accuracy: {test_class_acc[1]:.4f}")
print(f"  Non-Cracked Accuracy: {test_class_acc[0]:.4f}")

gap = train_acc - test_acc
print(f"\nTrain-Test Gap: {gap:.4f} {'(Overfitting)' if gap > 0.05 else '(Good generalization)'}")